# FraudScope — 03 Graph Neural Networks
## Phase 3 · Features graphe (IEEE-CIS) · GCN & GAT (Elliptic)

**Objectifs de ce notebook** :
1. Construire un **graphe de transactions** à partir du dataset IEEE-CIS avec NetworkX
2. Extraire des **features graphe** (degré, centralité, vélocité réseau) et mesurer leur apport sur XGBoost
3. Entraîner un **GCN** (Graph Convolutional Network) sur le dataset Elliptic Bitcoin
4. Entraîner un **GAT** (Graph Attention Network) sur le même dataset
5. Comparer GCN, GAT et le meilleur modèle Phase 2 (`FraudScopeXGB v2`) sur les métriques clés
6. Logger tous les runs dans MLflow (expérience `fraud-detection-paytrack`)

**Prérequis** :
- Avoir complété `01_exploration.ipynb` et `02_mlops.ipynb`
- Dataset Elliptic présent dans `data/elliptic/` (voir `data/get_dataset.md`)
- Serveur MLflow actif : `mlflow server --host 127.0.0.1 --port 8080 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlartifacts`

> ℹ️ Ce notebook utilise deux datasets distincts :
> - **IEEE-CIS** pour la partie features graphe + XGBoost (Partie A)
> - **Elliptic Bitcoin Dataset** pour l'entraînement GCN/GAT (Partie B)

## 0. Imports et configuration

In [6]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Graph (Partie A) ──────────────────────────────────────────
import networkx as nx

# ── ML classique ─────────────────────────────────────────────
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    recall_score, f1_score, average_precision_score,
    precision_recall_curve, roc_auc_score
)
import xgboost as xgb

# ── GNN (Partie B) ────────────────────────────────────────────
# Installation PyTorch Geometric :
# La commande varie selon la plateforme. Exemples :
#   CPU-only  : pip install torch-geometric
#   CUDA 11.8 : pip install torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
#   Colab     : pip install torch-geometric  (torch est déjà présent)
# Vérifier la compatibilité : https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch_geometric.utils import to_networkx

# ── MLflow ────────────────────────────────────────────────────
from dotenv import load_dotenv
import mlflow
import mlflow.pytorch
import mlflow.xgboost
from mlflow.tracking import MlflowClient

# ── Configuration ─────────────────────────────────────────────
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

FIGURES_DIR   = Path('figures')
ARTIFACTS_DIR = Path('artifacts')
FIGURES_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE   = 42
TARGET_IEEE    = 'isFraud'
DATA_DIR       = Path('data')
ELLIPTIC_DIR   = DATA_DIR / 'elliptic'
N_SPLITS_TSS   = 5
GRAPH_SAMPLE_N = 10_000
# Cap sur la taille des cliques lors de la construction du graphe.
# Évite l'explosion combinatoire sur les domaines génériques (ex: gmail.com).
# Un groupe de k cartes produit k*(k-1)/2 arêtes ; avec k=50 → 1225 arêtes max/groupe.
MAX_CLIQUE_SIZE = 50

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── MLflow setup ─────────────────────────────────────────────
load_dotenv()
TRACKING_URI    = os.getenv('MLFLOW_TRACKING_URI',    'http://127.0.0.1:8080')
EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'fraud-detection-paytrack')
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Device PyTorch : {DEVICE}')
print(f'MLflow URI     : {TRACKING_URI}')
print(f'Expérience     : {EXPERIMENT_NAME}')
import torch_geometric
print(f'PyG version    : {torch_geometric.__version__}')
print('Imports OK')

Device PyTorch : cpu
MLflow URI     : http://127.0.0.1:8080
Expérience     : fraud-detection-paytrack
PyG version    : 2.8.0
Imports OK


---
# PARTIE A — Features graphe sur IEEE-CIS + XGBoost

**Idée** : les fraudes ne sont pas des événements isolés — elles forment des clusters dans un réseau de transactions partageant des identifiants (cartes, adresses, e-mails, appareils). En modélisant ces relations comme un graphe, on extrait des signaux de structure réseau que les features tabulaires classiques ne capturent pas.

**Graphe construit** : chaque nœud est un identifiant de carte (`card1`) ; une arête relie deux cartes si elles ont effectué une transaction vers le même marchand (`P_emaildomain`) dans la même fenêtre temporelle (même journée). Les poids d'arête sont le nombre de co-occurrences.

## 1. Chargement IEEE-CIS et features Phase 1

In [7]:
assert (DATA_DIR / 'train_transaction.csv').exists(), "Données IEEE-CIS manquantes → voir data/get_dataset.md"

print('Chargement IEEE-CIS...')
train_tx = pd.read_csv(DATA_DIR / 'train_transaction.csv')
train_id = pd.read_csv(DATA_DIR / 'train_identity.csv')
df = train_tx.merge(train_id, on='TransactionID', how='left')
df = df.sort_values('TransactionDT').reset_index(drop=True)
print(f'Dataset : {df.shape[0]:,} transactions, {df.shape[1]} colonnes')

df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day']  = df['TransactionDT'] // (3600 * 24)
proxy_cols = [c for c in ['card1','card2','card3','card4','card5','card6','addr1','addr2'] if c in df.columns]
df['customer_proxy'] = df[proxy_cols].astype(str).fillna('NA').agg('_'.join, axis=1)
df['amount_ratio'] = df['TransactionAmt'] / (
    df.groupby('customer_proxy')['TransactionAmt'].transform('median').clip(lower=0.01)
)
df['amount_ratio'] = df['amount_ratio'].replace([np.inf, -np.inf], 1.0).fillna(1.0)
df['tx_rank_in_customer'] = df.groupby('customer_proxy').cumcount()
print('Features temporelles et vélocité OK')

Chargement IEEE-CIS...


MemoryError: Unable to allocate 1.49 GiB for an array with shape (339, 590540) and data type float64

## 2. Construction du graphe de transactions (NetworkX)

On travaille sur un **échantillon de `GRAPH_SAMPLE_N` transactions** pour la construction et la visualisation du graphe.

**Nœuds** : valeurs uniques de `card1` (identifiant de carte)
**Arêtes** : deux cartes sont reliées si elles ont la même `P_emaildomain` le même `day`
**Poids** : nombre de co-occurrences

In [ ]:
sample = df.head(GRAPH_SAMPLE_N).copy()
G = nx.Graph()

for _, row in sample.iterrows():
    node = str(row['card1'])
    if not G.has_node(node):
        G.add_node(node, is_fraud=int(row[TARGET_IEEE]))

grp = sample.groupby(['P_emaildomain', 'day'])['card1'].apply(list)
for cards in grp:
    cards = [str(c) for c in cards if pd.notna(c)]
    if len(cards) > MAX_CLIQUE_SIZE:
        cards = cards[:MAX_CLIQUE_SIZE]
    for i in range(len(cards)):
        for j in range(i+1, len(cards)):
            u, v = cards[i], cards[j]
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

print(f'Graphe : {G.number_of_nodes():,} noeuds (cartes), {G.number_of_edges():,} aretes')
print(f'Densite : {nx.density(G):.6f}')
print(f'Composantes connexes : {nx.number_connected_components(G):,}')

### 2.1 Visualisation d'un cluster suspect

In [ ]:
largest_cc = max(nx.connected_components(G), key=len)
G_sub = G.subgraph(largest_cc).copy()
print(f'Plus grande composante : {G_sub.number_of_nodes():,} noeuds, {G_sub.number_of_edges():,} aretes')

# Sous-graphe suspect : nœuds avec degree >= percentile 90
degrees = dict(G_sub.degree())
threshold = np.percentile(list(degrees.values()), 90)
suspect_nodes = [n for n, d in degrees.items() if d >= threshold]
G_suspect = G_sub.subgraph(suspect_nodes).copy()

fraud_nodes  = [n for n in G_suspect.nodes() if G_suspect.nodes[n].get('is_fraud', 0) == 1]
normal_nodes = [n for n in G_suspect.nodes() if G_suspect.nodes[n].get('is_fraud', 0) == 0]
print(f'Cluster suspect : {G_suspect.number_of_nodes()} noeuds ({len(fraud_nodes)} frauduleux)')

fig, ax = plt.subplots(figsize=(12, 8))
pos = nx.spring_layout(G_suspect, seed=RANDOM_STATE)
nx.draw_networkx_nodes(G_suspect, pos, nodelist=normal_nodes, node_color='steelblue',
                       node_size=50, alpha=0.6, ax=ax)
nx.draw_networkx_nodes(G_suspect, pos, nodelist=fraud_nodes, node_color='crimson',
                       node_size=120, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G_suspect, pos, alpha=0.2, ax=ax)
ax.set_title('Cluster suspect — rouge = frauduleux', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'graph_cluster_suspect.png', dpi=120)
plt.show()

## 3. Features graphe sur le dataset complet

On calcule maintenant les features de structure graphe sur l'ensemble des 590k transactions :
- **`graph_distinct_merchants_7d`** : vélocité inter-marchands sur 7 jours glissants
- **`graph_degree`** : degré du nœud dans G_full (nombre de cartes voisines)
- **`graph_betweenness`** : centralité betweenness (approximée, k=500)

In [ ]:
print('Calcul des features graphe...')
print('Calcul velocite inter-marchands (rolling 7j)...')

# ── Vélocité inter-marchands : sliding window O(n) par groupe ────────────────
# Algorithme deux-pointeurs (left/right) sur le groupe card1 trié par jour.
# Complexité : O(n log n) globale (dominée par le sort), O(k) mémoire par groupe
# (k = nb transactions dans le groupe, jamais le produit cartésien).
# Aucun merge cross-join → pas de MemoryError.

df_vel = (
    df[['card1', 'day', 'P_emaildomain']]
    .dropna(subset=['P_emaildomain'])
    .sort_values(['card1', 'day'])
    .reset_index(drop=True)
)

def _rolling_distinct_7d(group: pd.DataFrame) -> pd.Series:
    """Nombre de P_emaildomain distincts dans les 7 jours précédant chaque transaction.
    Implémentation O(n) par groupe via deux pointeurs (sliding window).
    Le groupe doit être trié par 'day' (garanti par le sort_values ci-dessus).
    """
    days    = group['day'].values
    domains = group['P_emaildomain'].values
    n       = len(days)
    result  = np.empty(n, dtype=np.float32)
    left    = 0
    window: dict = {}

    for right in range(n):
        d = str(domains[right])
        window[d] = window.get(d, 0) + 1
        # Expirer les entrées hors fenêtre (> 7 jours)
        while days[right] - days[left] > 7:
            old = str(domains[left])
            window[old] -= 1
            if window[old] == 0:
                del window[old]
            left += 1
        result[right] = float(len(window))

    return pd.Series(result, index=group.index)


df['graph_distinct_merchants_7d'] = (
    df_vel
    .groupby('card1', sort=False, group_keys=False)
    .apply(_rolling_distinct_7d)
    .reindex(df.index)
    .fillna(1.0)
    .astype(np.float32)
)

print(f"graph_distinct_merchants_7d — moyenne : {df['graph_distinct_merchants_7d'].mean():.3f}, "
      f"max : {df['graph_distinct_merchants_7d'].max():.0f}")
print('Features graphe OK')

## 4. Construction du graphe complet G_full (IEEE-CIS)

On construit maintenant le graphe sur l'ensemble du dataset pour en extraire les features de centralité.

In [ ]:
print('Construction de G_full...')
G_full = nx.Graph()
G_full.add_nodes_from(df['card1'].dropna().astype(str).unique())

grp_full = df.groupby(['P_emaildomain', 'day'])['card1'].apply(list)
for cards in grp_full:
    cards = [str(c) for c in cards if pd.notna(c)]
    if len(cards) > MAX_CLIQUE_SIZE:      # guard identique à la cellule sample
        cards = cards[:MAX_CLIQUE_SIZE]
    for i in range(len(cards)):
        for j in range(i+1, len(cards)):
            u, v = cards[i], cards[j]
            if G_full.has_edge(u, v):
                G_full[u][v]['weight'] += 1
            else:
                G_full.add_edge(u, v, weight=1)

print(f'G_full : {G_full.number_of_nodes():,} noeuds, {G_full.number_of_edges():,} aretes')
print(f'Densité : {nx.density(G_full):.8f}')

## 5. Features de centralité (G_full)

In [ ]:
print('Calcul degree centrality...')
degree_dict = dict(G_full.degree())
df['graph_degree'] = df['card1'].astype(str).map(degree_dict).fillna(0).astype(np.float32)

print('Calcul betweenness centrality (approximation stochastique k=500)...')
# k=500 : approximation nettement plus rapide qu un calcul exact O(VE)
# min(500, ...) évite une erreur si le graphe a moins de 500 noeuds
betweenness_dict = nx.betweenness_centrality(
    G_full, normalized=True, weight='weight',
    k=min(500, G_full.number_of_nodes())
)
df['graph_betweenness'] = df['card1'].astype(str).map(betweenness_dict).fillna(0).astype(np.float32)

print(f"graph_degree      — moyenne : {df['graph_degree'].mean():.3f}, max : {df['graph_degree'].max():.0f}")
print(f"graph_betweenness — moyenne : {df['graph_betweenness'].mean():.6f}, max : {df['graph_betweenness'].max():.6f}")
print('Centralité OK')

## 6. Entraînement XGBoost avec features graphe

On ajoute les trois features graphe (`graph_distinct_merchants_7d`, `graph_degree`, `graph_betweenness`) au jeu de features de Phase 2 et on entraîne un nouveau run XGBoost logué dans MLflow.

In [ ]:
GRAPH_FEATURES = ['graph_distinct_merchants_7d', 'graph_degree', 'graph_betweenness']
BASE_FEATURES  = [
    'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
    'C1','C2','C3','C4','C5','C6','C7','C8','C9','C10','C11','C12','C13','C14',
    'D1','D2','D3','D4','D5','D6','D7','D8','D9','D10','D11','D12','D13','D14','D15',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'V1','V2','V3','V4','V5','V6','V7','V8','V9','V10',
    'hour', 'day', 'amount_ratio', 'tx_rank_in_customer'
]
ALL_FEATURES = [f for f in BASE_FEATURES + GRAPH_FEATURES if f in df.columns]

cat_cols = [c for c in ALL_FEATURES if df[c].dtype == object]
num_cols = [c for c in ALL_FEATURES if c not in cat_cols]

X = df[ALL_FEATURES]
y = df[TARGET_IEEE]

tss = TimeSeriesSplit(n_splits=N_SPLITS_TSS)

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), cat_cols)
])

scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)

xgb_params = dict(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr', use_label_encoder=False,
    random_state=RANDOM_STATE, n_jobs=-1
)

cv_auprc, cv_recall, cv_f1 = [], [], []

with mlflow.start_run(run_name='FraudScopeXGB_v3_graph') as run:
    mlflow.log_params({**xgb_params, 'n_splits': N_SPLITS_TSS, 'graph_features': GRAPH_FEATURES})

    for fold, (train_idx, val_idx) in enumerate(tss.split(X)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        X_tr_p  = preprocessor.fit_transform(X_tr)
        X_val_p = preprocessor.transform(X_val)

        model = xgb.XGBClassifier(**xgb_params)
        model.fit(X_tr_p, y_tr,
                  eval_set=[(X_val_p, y_val)],
                  verbose=False)

        proba = model.predict_proba(X_val_p)[:, 1]
        pred  = (proba >= 0.5).astype(int)

        auprc  = average_precision_score(y_val, proba)
        recall = recall_score(y_val, pred, zero_division=0)
        f1     = f1_score(y_val, pred, zero_division=0)

        cv_auprc.append(auprc)
        cv_recall.append(recall)
        cv_f1.append(f1)

        mlflow.log_metrics({
            f'fold_{fold}_auprc':  auprc,
            f'fold_{fold}_recall': recall,
            f'fold_{fold}_f1':     f1
        })
        print(f'  Fold {fold} — AUPRC: {auprc:.4f}  Recall: {recall:.4f}  F1: {f1:.4f}')

    mean_auprc  = np.mean(cv_auprc)
    mean_recall = np.mean(cv_recall)
    mean_f1     = np.mean(cv_f1)

    mlflow.log_metrics({'mean_auprc': mean_auprc, 'mean_recall': mean_recall, 'mean_f1': mean_f1})
    mlflow.xgboost.log_model(model, artifact_path='model')

    print(f'\nFraudScopeXGB v3 (graph) — AUPRC: {mean_auprc:.4f}  Recall: {mean_recall:.4f}  F1: {mean_f1:.4f}')
    print(f'Run ID : {run.info.run_id}')

---
# PARTIE B — GCN & GAT sur Elliptic Bitcoin Dataset

Le dataset Elliptic contient 203 770 transactions Bitcoin labelisées (licit / illicit / unknown) représentées sous forme de graphe temporel. On entraîne deux architectures GNN :
- **GCN** (Graph Convolutional Network) — agrégation isotrope des voisins
- **GAT** (Graph Attention Network) — agrégation pondérée par attention multi-têtes

## 7. Chargement Elliptic

In [ ]:
assert ELLIPTIC_DIR.exists(), "Dataset Elliptic manquant → voir data/get_dataset.md"

print('Chargement Elliptic...')
ell_feat  = pd.read_csv(ELLIPTIC_DIR / 'elliptic_txs_features.csv',  header=None)
ell_edges = pd.read_csv(ELLIPTIC_DIR / 'elliptic_txs_edgelist.csv')
ell_class = pd.read_csv(ELLIPTIC_DIR / 'elliptic_txs_classes.csv')

ell_feat.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(1, ell_feat.shape[1]-1)]
ell_class.columns = ['txId', 'class']
ell_class['label'] = ell_class['class'].map({'1': 1, '2': 0, 'unknown': -1})

ell_df = ell_feat.merge(ell_class[['txId', 'label']], on='txId', how='left')
labeled = ell_df[ell_df['label'] != -1].reset_index(drop=True)

print(f'Transactions labelisées : {len(labeled):,}  (illicites : {labeled["label"].sum():,})')

# Mapping txId → index pour edge_index
id_map = {tx: i for i, tx in enumerate(labeled['txId'])}

src, dst = [], []
for _, row in ell_edges.iterrows():
    if row['txId1'] in id_map and row['txId2'] in id_map:
        src.append(id_map[row['txId1']])
        dst.append(id_map[row['txId2']])

edge_index = torch.tensor([src + dst, dst + src], dtype=torch.long)

feat_cols = [c for c in labeled.columns if c.startswith('f')]
x = torch.tensor(labeled[feat_cols].values, dtype=torch.float32)
y_ell = torch.tensor(labeled['label'].values, dtype=torch.long)

data_ell = Data(x=x, edge_index=edge_index, y=y_ell)
print(f'PyG Data : {data_ell}')

# Split temporel (80/20 sur time_step)
n = len(labeled)
train_mask = torch.zeros(n, dtype=torch.bool)
test_mask  = torch.zeros(n, dtype=torch.bool)
train_mask[:int(0.8*n)] = True
test_mask[int(0.8*n):]  = True
data_ell.train_mask = train_mask
data_ell.test_mask  = test_mask
print(f'Train : {train_mask.sum().item():,}  Test : {test_mask.sum().item():,}')

## 8. Définition des modèles GCN et GAT

In [ ]:
class GCNModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin   = nn.Linear(hidden_channels, out_channels)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.drop(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.drop(x)
        return self.lin(x)


class GATModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1,
                             concat=False, dropout=dropout)
        self.lin   = nn.Linear(hidden_channels, out_channels)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        x = self.drop(x)
        x = F.elu(self.conv2(x, edge_index))
        x = self.drop(x)
        return self.lin(x)


print('Architectures GCN et GAT définies')

## 9. Entraînement GCN + GAT

In [ ]:
def train_gnn(model, data, epochs=100, lr=1e-3, run_name='GNN'):
    model = model.to(DEVICE)
    data  = data.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    n_illicit = data.y[data.train_mask].sum().item()
    n_licit   = data.train_mask.sum().item() - n_illicit
    pos_weight = torch.tensor([n_licit / max(n_illicit, 1)], device=DEVICE)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    results = {}

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({'epochs': epochs, 'lr': lr, 'model': run_name})

        for epoch in range(1, epochs + 1):
            model.train()
            opt.zero_grad()
            out  = model(data.x, data.edge_index)
            loss = criterion(out[data.train_mask, 1].float(),
                             data.y[data.train_mask].float())
            loss.backward()
            opt.step()

            if epoch % 20 == 0:
                model.eval()
                with torch.no_grad():
                    logits = model(data.x, data.edge_index)
                    proba_t = torch.softmax(logits, dim=1)[:, 1]
                    pred_t  = logits.argmax(dim=1)

                    y_val = data.y[data.test_mask].cpu().numpy()
                    p_val = proba_t[data.test_mask].cpu().numpy()
                    d_val = pred_t[data.test_mask].cpu().numpy()

                    auprc  = average_precision_score(y_val, p_val)
                    recall = recall_score(y_val, d_val, zero_division=0)
                    f1     = f1_score(y_val, d_val, zero_division=0)

                    mlflow.log_metrics({'loss': loss.item(), 'auprc': auprc,
                                        'recall': recall, 'f1': f1}, step=epoch)
                    print(f'  Epoch {epoch:3d} — loss: {loss.item():.4f}  '
                          f'AUPRC: {auprc:.4f}  Recall: {recall:.4f}  F1: {f1:.4f}')

        results = {'auprc': auprc, 'recall': recall, 'f1': f1}
        mlflow.pytorch.log_model(model, artifact_path='model')

    return results


in_ch = data_ell.num_node_features
print(f'Features Elliptic : {in_ch}')

print('\n=== Entraînement GCN ===')
gcn_model = GCNModel(in_ch, hidden_channels=64, out_channels=2)
gcn_res   = train_gnn(gcn_model, data_ell, epochs=100, run_name='FraudScope_GCN')

print('\n=== Entraînement GAT ===')
gat_model = GATModel(in_ch, hidden_channels=32, out_channels=2, heads=4)
gat_res   = train_gnn(gat_model, data_ell, epochs=100, run_name='FraudScope_GAT')

## 10. Tableau comparatif final

In [ ]:
comparison = pd.DataFrame([
    {'Modèle': 'FraudScopeXGB v2 (Phase 2)',
     'Dataset': 'IEEE-CIS',
     'AUPRC': '—',
     'Recall': '—',
     'F1': '—',
     'Note': 'Référence Phase 2'},
    {'Modèle': 'FraudScopeXGB v3 (graph)',
     'Dataset': 'IEEE-CIS',
     'AUPRC': f'{mean_auprc:.4f}',
     'Recall': f'{mean_recall:.4f}',
     'F1': f'{mean_f1:.4f}',
     'Note': '+ features graphe'},
    {'Modèle': 'GCN',
     'Dataset': 'Elliptic',
     'AUPRC': f"{gcn_res['auprc']:.4f}",
     'Recall': f"{gcn_res['recall']:.4f}",
     'F1': f"{gcn_res['f1']:.4f}",
     'Note': 'Graph Convolutional Network'},
    {'Modèle': 'GAT',
     'Dataset': 'Elliptic',
     'AUPRC': f"{gat_res['auprc']:.4f}",
     'Recall': f"{gat_res['recall']:.4f}",
     'F1': f"{gat_res['f1']:.4f}",
     'Note': 'Graph Attention Network'},
])

print('\n=== Tableau comparatif Phase 3 ===')
print(comparison.to_string(index=False))
comparison.to_csv(ARTIFACTS_DIR / 'phase3_comparison.csv', index=False)
print('\nSauvegardé dans artifacts/phase3_comparison.csv')